In [0]:
%sql
WITH fleet AS (
  SELECT g.kommun_kod,
         SUM(CASE WHEN d.drivmedel = 'BEV'  AND g.ar = 2025 THEN g.antal END) AS bev_2025,
         SUM(CASE WHEN d.drivmedel = 'BEV'  AND g.ar = 2023 THEN g.antal END) AS bev_2023,
         SUM(CASE WHEN d.drivmedel = 'PHEV' AND g.ar = 2025 THEN g.antal END) AS phev_2025
  FROM laddstolpar_df.silver.trafa_t10026_grid g
  JOIN laddstolpar_df.silver.drivmedel d ON d.trafa_kod = g.drivmedel_kod
  GROUP BY g.kommun_kod
),
company AS (
  SELECT kommun_kod,
         SUM(CASE WHEN agarkategori = '030' THEN antal END)
           / NULLIF(SUM(CASE WHEN agarkategori = '000' THEN antal END), 0) AS company_share
  FROM laddstolpar_df.silver.scb_tab3276
  WHERE __END_AT IS NULL AND ar = 2025 AND enhet = 'antal'
  GROUP BY kommun_kod
),
chargers AS (
  SELECT s.kommun_kod,
         SUM(c.kw) AS public_kw,
         SUM(CASE WHEN c.current_type = 'DC' THEN c.kw END) AS public_dc_kw
  FROM laddstolpar_df.silver.nobil_station s
  JOIN laddstolpar_df.silver.nobil_connector c
    ON c.station_id = s.station_id AND c.__END_AT IS NULL
  WHERE s.__END_AT IS NULL AND s.is_public AND s.station_id <> 84436
    AND c.vehicle_type_code IN ('1','6','11','12','15','22')
  GROUP BY s.kommun_kod
),
price AS (
  SELECT elomrade, AVG(sek_per_kwh) AS avg_sek_12m, COUNT(DISTINCT datum_lokal) AS n_days
  FROM laddstolpar_df.silver.elpris
  WHERE datum_lokal > date_sub((SELECT MAX(datum_lokal) FROM laddstolpar_df.silver.elpris), 365)
  GROUP BY elomrade
),
k AS (
  SELECT km.kommun_kod, km.kommun_namn, km.elomrade, km.kommungrupp,
         f.bev_2025, f.bev_2025 - f.bev_2023 AS bev_growth_23_25,
         co.company_share,
         COALESCE(ch.public_kw, 0)    AS public_kw,
         COALESCE(ch.public_dc_kw, 0) AS public_dc_kw,
         1.3 * f.bev_2025 + 0.8 * f.phev_2025 AS afir_kw,
         p.avg_sek_12m
  FROM laddstolpar_df.silver.kommun km
  JOIN fleet f USING (kommun_kod)
  LEFT JOIN company  co USING (kommun_kod)
  LEFT JOIN chargers ch USING (kommun_kod)
  LEFT JOIN price    p  ON p.elomrade = km.elomrade
),
ranked AS (
  SELECT *, afir_kw - public_kw AS afir_gap_kw,
         RANK() OVER (ORDER BY bev_growth_23_25 DESC)   AS rk_growth,
         RANK() OVER (ORDER BY bev_2025 DESC)           AS rk_bev,
         RANK() OVER (ORDER BY afir_kw - public_kw DESC) AS rk_gap
  FROM k
)
SELECT 'kommun' AS section, rk_growth, kommun_kod, kommun_namn, elomrade, kommungrupp,
       bev_2025, bev_growth_23_25, rk_bev, ROUND(company_share, 3) AS company_share,
       ROUND(public_kw) AS public_kw, ROUND(public_dc_kw) AS public_dc_kw,
       ROUND(afir_kw) AS afir_kw, ROUND(afir_gap_kw) AS afir_gap_kw, rk_gap,
       ROUND(avg_sek_12m, 3) AS zone_avg_sek_12m, NULL AS n_days
FROM ranked WHERE rk_growth <= 20
UNION ALL
SELECT 'zone', NULL, NULL, 'Zone ' || elomrade, elomrade, NULL,
       NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL,
       ROUND(avg_sek_12m, 3), n_days
FROM price
ORDER BY section DESC, rk_growth, kommun_namn;

In [0]:
%sql
SELECT elomrade,
       ROUND(AVG(sek_per_kwh), 3) AS avg_sek,
       COUNT(DISTINCT datum_lokal) AS n_days
FROM laddstolpar_df.silver.elpris
GROUP BY elomrade
ORDER BY elomrade;

In [0]:
%sql
WITH fleet AS (
  SELECT g.kommun_kod,
         SUM(CASE WHEN d.drivmedel = 'BEV'  AND g.ar = 2025 THEN g.antal END) AS bev_2025,
         SUM(CASE WHEN d.drivmedel = 'BEV'  AND g.ar = 2023 THEN g.antal END) AS bev_2023,
         SUM(CASE WHEN d.drivmedel = 'PHEV' AND g.ar = 2025 THEN g.antal END) AS phev_2025
  FROM laddstolpar_df.silver.trafa_t10026_grid g
  JOIN laddstolpar_df.silver.drivmedel d ON d.trafa_kod = g.drivmedel_kod
  GROUP BY g.kommun_kod
),
company AS (
  SELECT kommun_kod,
         SUM(CASE WHEN agarkategori = '030' THEN antal END)
           / NULLIF(SUM(CASE WHEN agarkategori = '000' THEN antal END), 0) AS company_share
  FROM laddstolpar_df.silver.scb_tab3276
  WHERE __END_AT IS NULL AND ar = 2025 AND enhet = 'antal'
  GROUP BY kommun_kod
),
chargers AS (
  SELECT s.kommun_kod,
         SUM(c.kw) AS public_kw,
         SUM(CASE WHEN c.current_type = 'DC' THEN c.kw END) AS public_dc_kw
  FROM laddstolpar_df.silver.nobil_station s
  JOIN laddstolpar_df.silver.nobil_connector c
    ON c.station_id = s.station_id AND c.__END_AT IS NULL
  WHERE s.__END_AT IS NULL AND s.is_public AND s.station_id <> 84436
    AND c.vehicle_type_code IN ('1','6','11','12','15','22')
  GROUP BY s.kommun_kod
),
ranked AS (
  SELECT km.kommun_kod, km.kommun_namn, km.elomrade, km.kommungrupp,
         f.bev_2025,
         f.bev_2025 - f.bev_2023 AS bev_growth_23_25,
         co.company_share,
         COALESCE(ch.public_kw, 0)    AS public_kw,
         COALESCE(ch.public_dc_kw, 0) AS public_dc_kw,
         1.3 * f.bev_2025 + 0.8 * f.phev_2025 AS afir_kw,
         RANK() OVER (ORDER BY f.bev_2025 - f.bev_2023 DESC) AS rk_growth,
         RANK() OVER (ORDER BY f.bev_2025 DESC)              AS rk_bev,
         RANK() OVER (ORDER BY 1.3 * f.bev_2025 + 0.8 * f.phev_2025
                               - COALESCE(ch.public_kw, 0) DESC) AS rk_gap
  FROM laddstolpar_df.silver.kommun km
  JOIN fleet f USING (kommun_kod)
  LEFT JOIN company  co USING (kommun_kod)
  LEFT JOIN chargers ch USING (kommun_kod)
)
SELECT rk_growth, kommun_kod, kommun_namn, elomrade, kommungrupp,
       bev_2025, bev_growth_23_25, rk_bev,
       ROUND(company_share, 3)                 AS company_share,
       CAST(public_kw AS BIGINT)               AS public_kw,
       CAST(public_dc_kw AS BIGINT)            AS public_dc_kw,
       CAST(afir_kw AS BIGINT)                 AS afir_kw,
       CAST(afir_kw - public_kw AS BIGINT)     AS afir_gap_kw,
       rk_gap
FROM ranked
WHERE rk_growth <= 20
ORDER BY rk_growth;

In [0]:
%sql
WITH fleet AS (
  SELECT g.kommun_kod,
         SUM(CASE WHEN d.drivmedel = 'BEV'  AND g.ar = 2025 THEN g.antal END) AS bev_2025,
         SUM(CASE WHEN d.drivmedel = 'BEV'  AND g.ar = 2023 THEN g.antal END) AS bev_2023,
         SUM(CASE WHEN d.drivmedel = 'PHEV' AND g.ar = 2025 THEN g.antal END) AS phev_2025,
         SUM(CASE WHEN g.ar = 2025 THEN g.antal END) AS tot_2025,
         SUM(CASE WHEN g.ar = 2023 THEN g.antal END) AS tot_2023
  FROM laddstolpar_df.silver.trafa_t10026_grid g
  JOIN laddstolpar_df.silver.drivmedel d ON d.trafa_kod = g.drivmedel_kod
  GROUP BY g.kommun_kod
),
chargers AS (
  SELECT s.kommun_kod,
         SUM(c.kw) AS public_kw,
         SUM(CASE WHEN c.current_type = 'DC' THEN c.kw END) AS public_dc_kw
  FROM laddstolpar_df.silver.nobil_station s
  JOIN laddstolpar_df.silver.nobil_connector c
    ON c.station_id = s.station_id AND c.__END_AT IS NULL
  WHERE s.__END_AT IS NULL AND s.is_public AND s.station_id <> 84436
    AND c.vehicle_type_code IN ('1','6','11','12','15','22')
  GROUP BY s.kommun_kod
),
k AS (
  SELECT f.kommun_kod, f.bev_2025, f.phev_2025,
         COALESCE(ch.public_kw, 0)    AS public_kw,
         COALESCE(ch.public_dc_kw, 0) AS public_dc_kw,
         1.3 * f.bev_2025 + 0.8 * f.phev_2025 AS afir_kw,
         f.bev_2025 - f.bev_2023 AS growth_abs,
         f.bev_2025 / f.tot_2025 - f.bev_2023 / f.tot_2023 AS share_pp
  FROM laddstolpar_df.silver.kommun km
  JOIN fleet f USING (kommun_kod)
  LEFT JOIN chargers ch USING (kommun_kod)
),
r AS (
  SELECT *,
         RANK() OVER (ORDER BY bev_2025 DESC)             AS rk_bev,
         RANK() OVER (ORDER BY growth_abs DESC)           AS rk_growth_abs,
         RANK() OVER (ORDER BY share_pp DESC)             AS rk_share_pp,
         RANK() OVER (ORDER BY afir_kw - public_kw DESC)  AS rk_gap
  FROM k
)
SELECT COUNT(*)                          AS n_kommuner,
       COUNT_IF(afir_kw > public_kw)     AS n_gap_total_pos,
       COUNT_IF(afir_kw > public_dc_kw)  AS n_gap_dc_pos,
       SUM(bev_2025)                     AS bev_2025,
       SUM(phev_2025)                    AS phev_2025,
       CAST(SUM(afir_kw) AS BIGINT)      AS afir_kw,
       CAST(SUM(public_kw) AS BIGINT)    AS public_kw,
       CAST(SUM(public_dc_kw) AS BIGINT) AS public_dc_kw,
       ROUND(corr(rk_growth_abs, rk_bev), 2) AS rho_growth_abs_vs_bev,
       ROUND(corr(rk_share_pp,   rk_bev), 2) AS rho_share_pp_vs_bev,
       ROUND(corr(rk_gap,        rk_bev), 2) AS rho_gap_vs_bev
FROM r;

In [0]:
%sql
WITH reg AS (
  SELECT kommun_kod,
         SUM(CASE WHEN drivmedel_scb_kod = '120' THEN antal END) AS bev_new_12m,
         SUM(antal) AS all_new_12m
  FROM laddstolpar_df.silver.scb_tab3277
  WHERE __END_AT IS NULL
    AND period_start >= add_months((SELECT MAX(period_start) FROM laddstolpar_df.silver.scb_tab3277), -11)
  GROUP BY kommun_kod
),
fleet AS (
  SELECT g.kommun_kod,
         SUM(CASE WHEN d.drivmedel = 'BEV' AND g.ar = 2025 THEN g.antal END) AS bev_2025,
         SUM(CASE WHEN d.drivmedel = 'BEV' AND g.ar = 2023 THEN g.antal END) AS bev_2023,
         SUM(CASE WHEN g.ar = 2025 THEN g.antal END) AS tot_2025,
         SUM(CASE WHEN g.ar = 2023 THEN g.antal END) AS tot_2023
  FROM laddstolpar_df.silver.trafa_t10026_grid g
  JOIN laddstolpar_df.silver.drivmedel d ON d.trafa_kod = g.drivmedel_kod
  GROUP BY g.kommun_kod
),
company AS (
  SELECT kommun_kod,
         SUM(CASE WHEN agarkategori = '030' THEN antal END)
           / NULLIF(SUM(CASE WHEN agarkategori = '000' THEN antal END), 0) AS company_share
  FROM laddstolpar_df.silver.scb_tab3276
  WHERE __END_AT IS NULL AND ar = 2025 AND enhet = 'antal'
  GROUP BY kommun_kod
),
k AS (
  SELECT km.kommun_kod, km.kommun_namn, km.elomrade,
         f.bev_2025, f.bev_2023, r.all_new_12m, co.company_share,
         f.bev_2025 / f.tot_2025 - f.bev_2023 / f.tot_2023 AS share_pp,
         f.bev_2025 / NULLIF(f.bev_2023, 0)                AS rel_growth,
         r.bev_new_12m / NULLIF(r.all_new_12m, 0)          AS new_share_12m
  FROM laddstolpar_df.silver.kommun km
  JOIN fleet f   USING (kommun_kod)
  JOIN reg r     USING (kommun_kod)
  LEFT JOIN company co USING (kommun_kod)
),
r AS (
  SELECT *,
         RANK() OVER (ORDER BY bev_2025 DESC)      AS rk_bev,
         RANK() OVER (ORDER BY rel_growth DESC)    AS rk_rel,
         RANK() OVER (ORDER BY share_pp DESC)      AS rk_share_pp,
         RANK() OVER (ORDER BY new_share_12m DESC) AS rk_new_share
  FROM k
)
SELECT kommun_kod, kommun_namn, elomrade, bev_2025, rk_bev, all_new_12m,
       ROUND(company_share, 3)       AS company_share,
       ROUND(rel_growth, 2)          AS rel_growth,    rk_rel,
       ROUND(share_pp * 100, 2)      AS share_pp,      rk_share_pp,
       ROUND(new_share_12m * 100, 1) AS new_share_pct, rk_new_share,
       ROUND(corr(rk_rel,       rk_bev) OVER (), 2) AS rho_rel_vs_bev,
       ROUND(corr(rk_new_share, rk_bev) OVER (), 2) AS rho_new_share_vs_bev,
       ROUND(corr(rk_new_share, rk_share_pp) OVER (), 2) AS rho_new_share_vs_share_pp
FROM r
WHERE rk_rel <= 10 OR rk_share_pp <= 10 OR rk_new_share <= 10
ORDER BY rk_new_share;

In [0]:
%sql
WITH reg AS (
  SELECT t.kommun_kod, km.kommun_namn, km.lan_kod, km.elomrade,
         SUM(CASE WHEN t.drivmedel_scb_kod = '120' THEN t.antal END) AS bev_new,
         SUM(t.antal) AS all_new
  FROM laddstolpar_df.silver.scb_tab3277 t
  JOIN laddstolpar_df.silver.kommun km ON km.kommun_kod = t.kommun_kod
  WHERE t.__END_AT IS NULL
    AND t.period_start >= add_months((SELECT MAX(period_start) FROM laddstolpar_df.silver.scb_tab3277), -11)
  GROUP BY t.kommun_kod, km.kommun_namn, km.lan_kod, km.elomrade
),
county AS (
  SELECT lan_kod, SUM(bev_new) / SUM(all_new) AS p_county
  FROM reg GROUP BY lan_kod
),
nat AS (
  SELECT SUM(bev_new) / SUM(all_new) AS p_nat FROM reg
),
mom AS (
  SELECT AVG(POWER(r.bev_new / r.all_new - c.p_county, 2))
       - AVG(c.p_county * (1 - c.p_county) / r.all_new) AS tau2
  FROM reg r JOIN county c ON c.lan_kod = r.lan_kod
  WHERE r.all_new > 0
),
kk AS (
  SELECT n.p_nat, m.tau2, n.p_nat * (1 - n.p_nat) / m.tau2 - 1 AS k
  FROM nat n CROSS JOIN mom m
),
fleet AS (
  SELECT g.kommun_kod,
         SUM(CASE WHEN d.drivmedel = 'BEV'  AND g.ar = 2025 THEN g.antal END) AS bev_2025,
         SUM(CASE WHEN d.drivmedel = 'PHEV' AND g.ar = 2025 THEN g.antal END) AS phev_2025
  FROM laddstolpar_df.silver.trafa_t10026_grid g
  JOIN laddstolpar_df.silver.drivmedel d ON d.trafa_kod = g.drivmedel_kod
  GROUP BY g.kommun_kod
),
supply AS (
  SELECT s.kommun_kod,
         SUM(CASE WHEN c.current_type = 'DC' THEN c.kw END) AS public_dc_kw
  FROM laddstolpar_df.silver.nobil_station s
  JOIN laddstolpar_df.silver.nobil_connector c
    ON c.station_id = s.station_id AND c.__END_AT IS NULL
  WHERE s.__END_AT IS NULL AND s.is_public AND s.station_id <> 84436
    AND c.vehicle_type_code IN ('1','6','11','12','15','22')
  GROUP BY s.kommun_kod
),
m AS (
  SELECT r.kommun_kod, r.kommun_namn, r.elomrade, r.all_new,
         r.bev_new / NULLIF(r.all_new, 0)                        AS d_raw,
         (r.bev_new + kk.k * c.p_county) / (r.all_new + kk.k)    AS d_shrunk,
         c.p_county,
         f.bev_2025,
         COALESCE(s.public_dc_kw, 0)
           / NULLIF(f.bev_2025 + 0.6 * f.phev_2025, 0)           AS dc_kw_per_ev,
         kk.k, kk.tau2, kk.p_nat
  FROM reg r
  JOIN county c      ON c.lan_kod = r.lan_kod
  JOIN fleet f       ON f.kommun_kod = r.kommun_kod
  LEFT JOIN supply s ON s.kommun_kod = r.kommun_kod
  CROSS JOIN kk
),
ranked AS (
  SELECT *,
         RANK() OVER (ORDER BY d_raw DESC)        AS rk_d_raw,
         RANK() OVER (ORDER BY d_shrunk DESC)     AS rk_d_shrunk,
         RANK() OVER (ORDER BY bev_2025 DESC)     AS rk_bev,
         RANK() OVER (ORDER BY dc_kw_per_ev ASC)  AS rk_low_supply,
         percent_rank() OVER (ORDER BY d_shrunk)
           + percent_rank() OVER (ORDER BY dc_kw_per_ev DESC) AS look_index
  FROM m
),
final AS (
  SELECT *, RANK() OVER (ORDER BY look_index DESC) AS rk_look FROM ranked
)
SELECT rk_look, kommun_kod, kommun_namn, elomrade, bev_2025, rk_bev, all_new,
       ROUND(d_raw * 100, 1)    AS d_raw_pct,    rk_d_raw,
       ROUND(p_county * 100, 1) AS county_pct,
       ROUND(d_shrunk * 100, 1) AS d_shrunk_pct, rk_d_shrunk,
       ROUND(dc_kw_per_ev, 2)   AS dc_kw_per_ev, rk_low_supply,
       ROUND(look_index, 3)     AS look_index,
       ROUND(k, 0)              AS k,
       ROUND(tau2, 5)           AS tau2,
       ROUND(p_nat * 100, 1)    AS nat_pct
FROM final
WHERE rk_look <= 25 OR rk_d_raw <= 5
ORDER BY rk_look;

In [0]:
%sql
SELECT COUNT_IF(elomrade_secondary = '')      AS secondary_blank,
       COUNT_IF(elomrade_secondary IS NULL)   AS secondary_null,
       COUNT_IF(elomrade_is_split)            AS split_true,
       COUNT_IF(NOT elomrade_is_split)        AS split_false,
       COUNT_IF(elomrade_is_split IS NULL)    AS split_null,
       COUNT_IF(elomrade_basis = '')          AS basis_blank,
       COUNT_IF(elomrade_source = '')         AS source_blank
FROM laddstolpar_df.silver.kommun;

In [0]:
%sql
SELECT
  (SELECT COUNT(*)                   FROM laddstolpar_df.gold.dim_kommun)   AS kommun_rows,
  (SELECT COUNT(DISTINCT kommun_kod) FROM laddstolpar_df.gold.dim_kommun)   AS kommun_keys,
  (SELECT COUNT_IF(is_besoksnaring)  FROM laddstolpar_df.gold.dim_kommun)   AS besoksnaring,
  (SELECT COUNT(DISTINCT gruppkod)   FROM laddstolpar_df.gold.dim_kommun)   AS n_groups,
  (SELECT COUNT_IF(elomrade_is_split) FROM laddstolpar_df.gold.dim_kommun)  AS n_split,
  (SELECT concat_ws(', ', array_sort(collect_list(elomrade || '=' || elomrade_namn)))
     FROM laddstolpar_df.gold.dim_elomrade)                                  AS zones,
  (SELECT COUNT(*)                   FROM laddstolpar_df.gold.dim_drivmedel) AS drivmedel_rows,
  (SELECT COUNT(*)                   FROM laddstolpar_df.gold.dim_datum)     AS datum_rows,
  (SELECT MIN(datum)                 FROM laddstolpar_df.gold.dim_datum)     AS datum_min,
  (SELECT MAX(datum)                 FROM laddstolpar_df.gold.dim_datum)     AS datum_max,
  (SELECT COUNT(*) - COUNT(DISTINCT datum) FROM laddstolpar_df.gold.dim_datum) AS datum_dups,
  (SELECT COUNT(DISTINCT scb_period) FROM laddstolpar_df.gold.dim_datum)     AS n_months,
  (SELECT COUNT(*) FROM laddstolpar_df.silver.scb_tab3277 s
     LEFT ANTI JOIN (SELECT DISTINCT scb_period FROM laddstolpar_df.gold.dim_datum) d
       ON d.scb_period = s.period
     WHERE s.__END_AT IS NULL)                                               AS tab3277_rows_without_date;

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM laddstolpar_df.gold.fct_nyregistrering_manad)                                  AS nyreg_rows,
  (SELECT COUNT(DISTINCT kommun_kod, scb_period, drivmedel) FROM laddstolpar_df.gold.fct_nyregistrering_manad) AS nyreg_keys,
  (SELECT SUM(antal) FROM laddstolpar_df.gold.fct_nyregistrering_manad)
    - (SELECT SUM(antal) FROM laddstolpar_df.silver.scb_tab3277 WHERE __END_AT IS NULL)               AS nyreg_sum_diff,
  (SELECT COUNT(*) FROM laddstolpar_df.gold.fct_fordon_i_trafik_ar)                                    AS fordon_rows,
  (SELECT antal FROM laddstolpar_df.gold.fct_fordon_i_trafik_ar
     WHERE kommun_kod = '0180' AND ar = 2025 AND drivmedel = 'BEV')                                    AS sthlm_bev_2025,
  (SELECT COUNT(*) FROM laddstolpar_df.gold.fct_elpris_dag)                                            AS elpris_dag_rows,
  (SELECT COUNT_IF(n_kvartar NOT IN (92, 96, 100)) FROM laddstolpar_df.gold.fct_elpris_dag)            AS elpris_bad_days,
  (SELECT ROUND(AVG(sek_medel), 3) FROM laddstolpar_df.gold.fct_elpris_dag WHERE elomrade = 'SE3')     AS se3_mean_of_days,
  (SELECT COUNT(*) FROM laddstolpar_df.gold.fct_kommun_ar)                                             AS kommun_ar_rows,
  (SELECT COUNT(DISTINCT kommun_kod, ar) FROM laddstolpar_df.gold.fct_kommun_ar)                       AS kommun_ar_keys,
  (SELECT SUM(folkmangd) FROM laddstolpar_df.gold.fct_kommun_ar WHERE ar = 2025)                       AS pop_2025,
  (SELECT andel_juridisk FROM laddstolpar_df.gold.fct_kommun_ar WHERE kommun_kod = '0184' AND ar = 2025) AS solna_juridisk_2025,
  (SELECT COUNT_IF(folkmangd IS NULL OR personbilar_totalt IS NULL) FROM laddstolpar_df.gold.fct_kommun_ar) AS kommun_ar_gaps;

In [0]:
%sql
WITH f AS (SELECT * FROM laddstolpar_df.gold.fct_laddstation)
SELECT
  (SELECT COUNT(*) FROM f)                                     AS stations,
  (SELECT COUNT(DISTINCT station_id) FROM f)                   AS station_keys,
  (SELECT COUNT(*) FROM laddstolpar_df.silver.nobil_station
     WHERE __END_AT IS NULL AND is_public AND station_id <> 84436)
    - (SELECT COUNT(*) FROM f)                                 AS public_without_carvan_power,
  (SELECT COUNT_IF(station_id = 84436) FROM f)                 AS test_station,
  (SELECT SUM(n_anslutningar) FROM f)                          AS connectors,
  (SELECT SUM(n_evse) FROM f)                                  AS evses,
  (SELECT CAST(SUM(kw_total) AS BIGINT) FROM f)                AS kw_total,
  (SELECT CAST(SUM(kw_dc) AS BIGINT) FROM f)                   AS kw_dc,
  (SELECT COUNT_IF(is_fast) FROM f)                            AS fast_stations,
  (SELECT COUNT_IF(is_ultra) FROM f)                           AS ultra_stations,
  (SELECT COUNT_IF(NOT position_valid) FROM f)                 AS invalid_positions,
  (SELECT COUNT(DISTINCT kommun_kod) FROM f)                   AS kommuner_with_station,
  (SELECT COUNT(DISTINCT kommun_kod) FROM f WHERE kw_dc > 0)   AS kommuner_with_dc;

In [0]:
%sql
SELECT c.gruppkod,
       c.kommungrupp,
       c.seed_kommungrupp,
       c.cost_factor,
       c.kommungrupp = c.seed_kommungrupp                 AS label_ok,
       length(c.kommungrupp)                              AS len_dim,
       length(c.seed_kommungrupp)                         AS len_seed,
       (SELECT COUNT(*) FROM laddstolpar_df.gold.dim_kommun k
         WHERE k.gruppkod = c.gruppkod)                   AS n_kommuner
FROM laddstolpar_df.gold.param_cost_factor c
ORDER BY c.gruppkod;

In [0]:
for t in ["seed_param_scenario_weights", "seed_param_variant", "seed_param_cost_factor", "seed_param_model"]:
    print(t, [repr(c) for c in spark.table(f"laddstolpar_df.bronze.{t}").columns])

In [0]:
%sql
WITH m AS (SELECT * FROM laddstolpar_df.gold.mart_kommun_metric)
SELECT
  (SELECT COUNT(*) FROM laddstolpar_df.gold.param_scenario_weights)                                        AS weight_rows,
  (SELECT MAX(ABS(s - 1)) FROM (SELECT SUM(weight) AS s FROM laddstolpar_df.gold.param_scenario_weights GROUP BY scenario)) AS max_sum_error,
  (SELECT COUNT(*) FROM laddstolpar_df.gold.param_cost_factor)                                             AS cost_groups,
  (SELECT COUNT(*) FROM laddstolpar_df.gold.param_variant)                                                 AS variants,
  (SELECT COUNT(*) FROM m)                                                                                  AS metric_rows,
  (SELECT COUNT(DISTINCT metric) FROM m)                                                                    AS n_metrics,
  (SELECT ROUND(MAX(k), 1) FROM m WHERE metric = 'growth')                                                  AS k,
  (SELECT MAX(detail) FROM m WHERE metric = 'growth' AND kommun_kod = '0180')                               AS sthlm_growth_detail,
  (SELECT ROUND(100 * value, 1) FROM m WHERE metric = 'growth_raw' AND kommun_kod = '2463')                 AS asele_raw_pct,
  (SELECT ROUND(100 * value, 1) FROM m WHERE metric = 'growth'     AND kommun_kod = '2463')                 AS asele_shrunk_pct,
  (SELECT ROUND(value, 3) FROM m WHERE metric = 'price' AND kommun_kod = '0180')                            AS se3_price,
  (SELECT COUNT_IF(value = 0) FROM m WHERE metric = 'supply')                                               AS zero_dc_kommuner,
  (SELECT ROUND(SUM(numerator) / SUM(denominator), 2) FROM m WHERE metric = 'supply')                       AS national_dc_kw_per_ev,
  (SELECT COUNT_IF(a.value <> b.value) FROM m a JOIN m b USING (kommun_kod)
     WHERE a.metric = 'price_zone_alt' AND b.metric = 'price')                                              AS zone_alt_changed,
  (SELECT ROUND(corr(a.pct_rank, b.pct_rank), 2) FROM m a JOIN m b USING (kommun_kod)
     WHERE a.metric = 'growth' AND b.metric = 'ev_demand')                                                  AS rho_growth_demand,
  (SELECT ROUND(corr(a.pct_rank, b.pct_rank), 2) FROM m a JOIN m b USING (kommun_kod)
     WHERE a.metric = 'supply' AND b.metric = 'ev_demand')                                                  AS rho_supply_demand;